- We have seen that ablating the attention mechanism hinders performance on both NounPP on short term memory. However the attention patterns of the attention head don't focus neither on the subject nor the attractor. Still, in relative terms the attention weights for the subject are more impoortant than those on the attractor.

- Does the difference between the attention weight of the subject and the one of the attractor correlate with performance on NounPP ?

- Hyp : If attention encodes long range subject verb agreement in a forced choice task, then only the relative weights for subject and attractor should matter for the task. If the difference correlates it could be the case, otherwise it is not.

- The differences in attention weights are normalized by the temperature during the test. Indeed, as training progresses, the attention decreases, and thus makes differences vanish

# 1. extracting attention weights and accuracy through time

1. Loop over checkpoints
2. For each checkpoint, load the model and evaluate it on NounPP while saving attention weights
3. Extract median attention weights for both subject and attractor + accuracy
4. compute difference and store it in a dictionary

In [76]:
import sys
import os

sys.path.append('/scratch2/mrenaudin/colorlessgreenRNNs')

import torch
from wm_tests.utils import CBR_RNN_attn_tracking
from evaluation_notebooks.utils import NounPPDataset, collate_fn_nounpp
from src.language_models.dictionary_corpus import Dictionary
from pathlib import Path
from torch.utils.data import DataLoader
from collections import defaultdict
from tqdm import tqdm
from scipy.stats import spearmanr


In [71]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CBR_RNN_attn_tracking(50001, 650, 650, 1, 0, device)
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
nounpp = "//scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.txt"
checkpoint_dir_str = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/test_attention_650"
checkpoint_files =  [f'epoch_{i}.pt' for i in range(1, 39, 1)]
checkpoint_dir = Path(checkpoint_dir_str)

In [72]:
test_dataset = NounPPDataset(nounpp, dictionary)
test_dataloader = DataLoader(test_dataset, batch_size=1000, collate_fn=collate_fn_nounpp)


In [125]:
def eval(model, test_dataloader, temperature, nheads, gumbel_softmax):
    condition_accuracies = defaultdict(int)
    condition_counts = defaultdict(int)
    correct_pred = 0
    sentence_details = []
    model.eval()
    rel_attn = {}
    # Forward pass with hidden state update word by word
    with torch.no_grad():
        for batch in test_dataloader:
            out = None
            written = batch["sentence"]
            sentence = batch["encoded_sentence"]
            correct = batch["encoded_correct"]
            wrong = batch["encoded_wrong"]
            condition = batch["condition"]
            c = condition[0] #with a batch size of 1000, 1 condition per batch
            batch_size = sentence.size(0)

            sent = sentence[:, :5].transpose(0, 1)
            cache = model.init_cache(sent,1)  # regarder si on peut mettre du priming
            # for i in range(sent.shape[1]):
            out, cache, attention_weights = model(sent, cache, nheads, temperature, gumbel_softmax)
            
            log_probs = torch.nn.functional.log_softmax(
                out, dim=-1
            )  # s(out.squeeze(0))
            # déja sur correct et wrong log probs, pas les même résultats que sur extract_predictions.py
            correct_log_probs = log_probs[
                -1, torch.arange(batch_size), correct
            ]  # Shape: [512]
            wrong_log_probs = log_probs[-1, torch.arange(batch_size), wrong]
            correct_predictions = correct_log_probs >= wrong_log_probs

            for i in range(batch_size):
                cond = condition[i]
                pred = correct_predictions[i].item()  # Convert tensor to Python boolean
                condition_counts[cond] += 1
                condition_accuracies[cond] += pred

                sentence_details.append(
                    {
                        "sentence": written[i],
                        "condition": condition[i],
                        "correct_log_prob": correct_log_probs[i],
                        "wrong_log_prob": wrong_log_probs[i],
                        "model_prefers_correct": pred,
                    }
                )
            sq = attention_weights[4].squeeze()#take attention when predicting the verb
            subject = sq[:,1]
            attractor = sq[:,4]
            diff = (subject-attractor)/temperature
            rel_attn[f'{c}']=diff.mean()

    final_accuracies = {
        cond: condition_accuracies[cond] / condition_counts[cond]
        for cond in condition_accuracies
    }
    return final_accuracies, rel_attn


In [126]:
test={}
test['acc']={}
test['relative_weights']={}

for item_name in tqdm(checkpoint_files, desc="Processing checkpoints"):
    item_path = checkpoint_dir / item_name
    checkpoint = torch.load(item_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    temperature = checkpoint['epoch'] 
    epoch = checkpoint['temperature']
    accuracy, relative_attention = eval(model, test_dataloader, temperature=temperature, nheads=1, gumbel_softmax=True)
    test['acc'][epoch]= accuracy
    test['relative_weights'][epoch]=relative_attention
  

Processing checkpoints: 100%|██████████| 38/38 [03:19<00:00,  5.25s/it]


In [129]:
test['acc'][1].keys()
for condition in test['acc'][1]:
    acc = [test['acc'][i][condition] for i in test['acc']]
    rel_attn = [test['relative_weights'][i][condition]for i in test['acc']]
    corr, pval = spearmanr(acc, rel_attn)
    print(f"Spearman correlation between median_repeat and '{condition}': {corr:.3f} (p={pval:.4f})")
    

Spearman correlation between median_repeat and 'singular singular': 0.160 (p=0.3362)
Spearman correlation between median_repeat and 'singular plural': -0.231 (p=0.1635)
Spearman correlation between median_repeat and 'plural singular': -0.103 (p=0.5384)
Spearman correlation between median_repeat and 'plural plural': -0.263 (p=0.1108)


Results are not conclusive, so it may not be the right direction. 
